# 실험 제목: 스트리밍 출력 (Streaming)

**목표**: ChatGPT처럼 답변이 한 글자씩 실시간으로 출력되도록 `stream=True` 옵션을 사용해보고, UX 측면에서의 장점을 확인합니다.

In [ ]:
# 필요한 라이브러리 import
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
# .env 파일 로드 및 OpenAI 클라이언트 생성
load_dotenv()
client = OpenAI()

### 1. 일반적인 호출 방식 (기다림)
모든 답변이 완성될 때까지 기다렸다가 한 번에 출력합니다.

In [ ]:
prompt = "인공지능의 미래에 대해 3문장으로 설명해줘."

print("[일반 호출 시작 - 기다리는 중...]")
start_time = time.time()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

end_time = time.time()
print(f"\n[출력 완료] (소요 시간: {end_time - start_time:.2f}초)")
print(response.choices[0].message.content)

### 2. 스트리밍(Streaming) 방식
답변이 생성되는 즉시 작은 덩어리(Chunk) 단위로 받아와서 출력합니다.

In [ ]:
print("[스트리밍 호출 시작]")
start_time = time.time()

# stream=True 옵션을 추가합니다.
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    stream=True
)

print("AI 답변: ", end="")

# 데이터가 들어오는 대로 실시간 출력
for chunk in stream:
    # chunk.choices[0].delta.content 에 글자 조각이 담겨 있습니다.
    content = chunk.choices[0].delta.content
    if content is not None:
        print(content, end="", flush=True)  # flush=True를 해야 버퍼에 쌓이지 않고 즉시 출력됨

end_time = time.time()
print(f"\n\n[출력 완료] (총 소요 시간: {end_time - start_time:.2f}초)")

### 실험 결과 정리

* **일반 호출**: 답변 생성이 전부 끝날 때까지 대기해야 하므로, 답변이 길어질 경우 사용자는 시스템이 멈춘 것으로 오해할 수 있습니다.
* **스트리밍**: `stream=True`를 사용하면 글자가 하나씩 타자 쳐지듯 출력되어 체감 응답 속도가 훨씬 빠르며, UX를 크게 개선할 수 있음을 확인했습니다.

### Notion 실험 로그 기록용

* **결과**: `stream=True` 옵션과 반복문을 사용하여 실시간 응답 출력을 성공적으로 구현함.
* **문제점**: 스트리밍 중에는 전체 토큰 사용량을 한 번에 파악하기 어렵고, 네트워크 상태에 따라 출력이 끊길 수 있음.
* **다음 개선 방향**: LangChain 적용 시 스트리밍 출력을 어떻게 활성화(Streaming Callback 등)하는지 학습해 볼 예정.